# Part 4 (1/2) — Category Data Preparation — **run on Kaggle**

This notebook does **all the data work and makes zero calls to the Metis
API**: download the dataset, run the analysis for **every eligible category**
(not just the top one), compute the quantitative stats, and package everything the LLM steps will need into a
single file, `category_prep_data.json`.

**Why this notebook is separate:** the Metis API (`api.metisai.ir`) is only
reachable from your local machine, not from Kaggle. So — exactly like
`digikala6_clean.ipynb` (data/embeddings on Kaggle) feeds
`run_validation.ipynb` (API calls, local) in the earlier parts of this
project — this notebook does the heavy data work here on Kaggle, and
`category_analysis_local.ipynb` does the API calls on your machine.

**Note on GPU:** unlike Part 1's RAG notebook, nothing here needs a GPU or
an embedding model — this is plain pandas aggregation plus preparing text
batches for the LLM. Kaggle is still convenient for downloading and
scanning the full comments CSV without tying up your local machine, but a
free CPU-only Kaggle session is enough.

**What to do with the output:** after running this notebook top to bottom,
download `category_prep_data.json` from Kaggle's output files and put it
next to `category_analysis_local.ipynb` on your local machine.


## بخش ۱ — نصب و ایمپورت

In [ ]:
import json

import pandas as pd
import numpy as np

from huggingface_hub import hf_hub_download
import matplotlib.pyplot as plt


## بخش ۲ — تنظیمات

In [ ]:
REPO_ID = "RadeAI/Digikala_comments_products"
REVISION = "89c3133b169c8d3793db8834f56f32fee33d9db0"

COMMENTS_FILE = "digikala-comments.csv"
PRODUCTS_FILE = "digikala-products.csv"

CATEGORY_COLUMN = "Category1"            # ستونی که دسته بر اساس آن انتخاب می‌شود
MIN_PRODUCTS_PER_CATEGORY = 5            # حداقل تعداد محصول تا یک دسته «واجد شرایط تحلیل» باشد (مقایسه‌ی برند معنادار شود)
MAX_COMMENTS_FOR_CATEGORY = 20_000       # سقف نظرات بارگذاری‌شده برای هر دسته (مدیریت حافظه)
MAX_COMMENTS_FOR_THEME_EXTRACTION = 800  # سقف نظرات منفی هر دسته برای مرحله‌ی Map-Reduce (اجرا می‌شود در نوت‌بوک محلی)
BATCH_SIZE = 40                          # اندازه‌ی هر Batch در مرحله‌ی Map
TOP_N_BRANDS = 4                         # تعداد برند اصلی برای مقایسه، به‌ازای هر دسته
SAMPLE_COMMENTS_PER_BRAND = 6            # تعداد نمونه نظر (مثبت+منفی) برای هر برند

HIGH_VOLUME_PERCENTILE = 0.67            # آستانه‌ی "پرنظر" (بالای این صدک) — داخل هر دسته محاسبه می‌شود
LOW_RECOMMENDATION_PERCENTILE = 0.33     # آستانه‌ی "نامطلوب" (پایین این صدک) — داخل هر دسته محاسبه می‌شود
MIN_COMMENTS_FOR_RATE = 10               # حداقل نظر لازم تا نرخ پیشنهاد یک محصول معنادار باشد

RANDOM_STATE = 42

OUTPUT_PREP_JSON = "category_prep_data.json"   # این فایل را از Kaggle دانلود و به محیط local منتقل کنید


## بخش ۳ — دانلود دیتاست

In [ ]:
comments_path = hf_hub_download(repo_id=REPO_ID, filename=COMMENTS_FILE, repo_type="dataset", revision=REVISION)
products_path = hf_hub_download(repo_id=REPO_ID, filename=PRODUCTS_FILE, repo_type="dataset", revision=REVISION)

df_products = pd.read_csv(products_path)
print("Products shape:", df_products.shape)


## بخش ۴ — انتخاب همه‌ی دسته‌های واجد شرایط (Eligible Categories)

به‌جای انتخاب فقط پرنظرترین دسته، **همه‌ی دسته‌هایی که حداقل
`MIN_PRODUCTS_PER_CATEGORY` محصول دارند** (تا مقایسه‌ی برند در ادامه معنادار
باشد) به‌عنوان دسته‌ی هدف در نظر گرفته می‌شوند. این لیست **hardcode نیست** —
با هر بار اجرا روی دیتاست واقعی محاسبه می‌شود؛ اگر گروه فقط چند دسته‌ی
مشخص مدنظرش است، کافی است `TARGET_CATEGORIES` را در پایین این سلول به‌صورت
دستی محدود کند (مثلاً `TARGET_CATEGORIES = TARGET_CATEGORIES[:5]`).


In [ ]:
chunksize = 200_000
comment_counts = {}

for chunk in pd.read_csv(comments_path, usecols=["id", "product_id"], chunksize=chunksize):
    counts = chunk["product_id"].value_counts()
    for product_id, count in counts.items():
        comment_counts[product_id] = comment_counts.get(product_id, 0) + int(count)

df_products["comment_count"] = df_products["id"].map(comment_counts).fillna(0).astype(int)

category_stats = (
    df_products.groupby(CATEGORY_COLUMN)
    .agg(total_comments=("comment_count", "sum"), product_count=("id", "count"))
    .sort_values("total_comments", ascending=False)
)

eligible_categories = category_stats[category_stats["product_count"] >= MIN_PRODUCTS_PER_CATEGORY]
TARGET_CATEGORIES = list(eligible_categories.index)

print(f"Eligible categories (>= {MIN_PRODUCTS_PER_CATEGORY} products): {len(TARGET_CATEGORIES)} از {len(category_stats)} دسته‌ی کل")
display(category_stats.head(10))


## بخش ۵ — بارگذاری یک‌بارهٔ نظرات همه‌ی دسته‌های واجد شرایط

به‌جای اسکن مجدد کل فایل نظرات برای هر دسته (که با ده‌ها دسته بسیار کند
می‌شد)، فایل نظرات فقط **یک‌بار** خوانده می‌شود و نظرات مربوط به محصولاتِ
هر دسته‌ی واجد شرایط (اجتماع همه‌ی آن‌ها) نگه داشته می‌شوند. تفکیک به‌ازای
هر دسته و اعمال سقف `MAX_COMMENTS_FOR_CATEGORY` در بخش بعد، داخل حلقه، انجام
می‌شود.


In [ ]:
all_eligible_product_ids = set(
    df_products[df_products[CATEGORY_COLUMN].isin(TARGET_CATEGORIES)]["id"]
)
print("Products across all eligible categories:", len(all_eligible_product_ids))

required_columns = [
    "id", "title", "body", "rate", "recommendation_status", "product_id",
    "advantages", "disadvantages", "likes", "dislikes",
]

all_comments_parts = []
for chunk in pd.read_csv(comments_path, usecols=required_columns, chunksize=chunksize):
    filtered = chunk[chunk["product_id"].isin(all_eligible_product_ids)]
    if len(filtered) > 0:
        all_comments_parts.append(filtered)

df_all_comments = pd.concat(all_comments_parts, ignore_index=True)
print("Total comments loaded across all eligible categories:", len(df_all_comments))


## بخش ۶ — توابع تحلیل (اجرا می‌شوند به‌ازای هر دسته)

منطق تحلیل کمّی، شناسایی محصولات «پرنظر ولی نامطلوب»، تحلیل به‌تفکیک برند،
و آماده‌سازی Batchهای نظرات منفی — همگی داخل تابع `analyze_category` جمع
شده‌اند تا در حلقه‌ی بخش بعد، بدون تغییر، روی تک‌تک دسته‌ها اجرا شوند. خروجیِ
هر دسته دقیقاً همان اسکیمایی را دارد که پیش‌تر برای یک دسته تولید می‌شد
(`category`, `products_analyzed`, `high_volume_low_recommendation_products`,
`top_brands_stats`, `brand_samples`, `negative_comment_batches`, ...) — چیزی
در فرم آن تغییر نکرده، فقط حالا برای هر دسته یک نمونه از آن ساخته می‌شود.


In [ ]:
def compute_recommendation_rate(group):
    rec = (group["recommendation_status"] == "recommended").sum()
    not_rec = (group["recommendation_status"] == "not_recommended").sum()
    total_decided = rec + not_rec
    if total_decided == 0:
        return np.nan
    return rec / total_decided


def sample_brand_comments(category_products, df_comments, brand, n=SAMPLE_COMMENTS_PER_BRAND):
    brand_product_ids = category_products[category_products["Brand"] == brand]["id"]
    brand_comments = df_comments[df_comments["product_id"].isin(brand_product_ids)]

    positive = brand_comments[brand_comments["recommendation_status"] == "recommended"].sort_values("likes", ascending=False).head(n // 2)
    negative = brand_comments[brand_comments["recommendation_status"] == "not_recommended"].sort_values("likes", ascending=False).head(n - len(positive))

    samples = []
    for _, row in pd.concat([positive, negative]).iterrows():
        samples.append({
            "comment_id": int(row["id"]),
            "sentiment": "positive" if row["recommendation_status"] == "recommended" else "negative",
            "snippet": f"{row['title']} — {row['body']}"[:300],
        })
    return samples


def analyze_category(category_name, category_products, df_comments):
    """همان تحلیلی که قبلاً فقط برای TARGET_CATEGORY انجام می‌شد، اکنون به‌ازای
    هر دسته فراخوانی می‌شود. در صورت نبود داده‌ی کافی، None برمی‌گرداند."""

    category_product_ids = set(category_products["id"])

    df_cat_comments = df_comments[df_comments["product_id"].isin(category_product_ids)].copy()
    if len(df_cat_comments) > MAX_COMMENTS_FOR_CATEGORY:
        df_cat_comments = df_cat_comments.sample(n=MAX_COMMENTS_FOR_CATEGORY, random_state=RANDOM_STATE)

    if len(df_cat_comments) == 0:
        return None

    # --- تحلیل کمّی: نرخ پیشنهاد خرید به تفکیک محصول ---
    product_stats = df_cat_comments.groupby("product_id").agg(
        comment_count=("id", "count"),
        avg_rate=("rate", "mean"),
    )
    product_stats["recommendation_rate"] = df_cat_comments.groupby("product_id").apply(
        compute_recommendation_rate, include_groups=False
    )
    product_stats = product_stats.merge(
        category_products[["id", "title_fa", "Brand"]], left_index=True, right_on="id"
    )
    product_stats = product_stats[product_stats["comment_count"] >= MIN_COMMENTS_FOR_RATE].copy()

    if len(product_stats) == 0:
        return None

    # --- محصولات «پرنظر ولی نامطلوب» ---
    volume_threshold = product_stats["comment_count"].quantile(HIGH_VOLUME_PERCENTILE)
    recommendation_threshold = product_stats["recommendation_rate"].quantile(LOW_RECOMMENDATION_PERCENTILE)

    flagged = product_stats[
        (product_stats["comment_count"] >= volume_threshold)
        & (product_stats["recommendation_rate"] <= recommendation_threshold)
    ].sort_values("comment_count", ascending=False)

    # --- تحلیل به تفکیک برند ---
    brand_stats = (
        product_stats.groupby("Brand")
        .agg(
            comment_count=("comment_count", "sum"),
            recommendation_rate=("recommendation_rate", "mean"),
            avg_rate=("avg_rate", "mean"),
            product_count=("id", "count"),
        )
        .sort_values("comment_count", ascending=False)
    )
    top_brands = brand_stats.head(TOP_N_BRANDS)

    # --- Batchهای نظرات منفی برای مرحله‌ی Map (اجرا در نوت‌بوک local) ---
    negative_comments = df_cat_comments[
        (df_cat_comments["recommendation_status"] == "not_recommended")
        | (df_cat_comments["disadvantages"].notna() & (df_cat_comments["disadvantages"].astype(str).str.len() > 3))
    ].copy()
    if len(negative_comments) > MAX_COMMENTS_FOR_THEME_EXTRACTION:
        negative_comments = negative_comments.sample(n=MAX_COMMENTS_FOR_THEME_EXTRACTION, random_state=RANDOM_STATE)
    negative_comments = negative_comments.reset_index(drop=True)

    negative_fields = ["id", "title", "body", "disadvantages", "recommendation_status"]
    negative_records = negative_comments[negative_fields].to_dict(orient="records")
    negative_comment_batches = [
        negative_records[i:i + BATCH_SIZE] for i in range(0, len(negative_records), BATCH_SIZE)
    ]

    # --- نمونه نظرات هر برند ---
    brand_samples = {
        brand: sample_brand_comments(category_products, df_cat_comments, brand)
        for brand in top_brands.index
    }

    prep_data = {
        "category": category_name,
        "products_analyzed": int(len(product_stats)),
        "comments_analyzed_for_themes": int(len(negative_comments)),
        "high_volume_low_recommendation_products": json.loads(
            flagged[["title_fa", "Brand", "comment_count", "recommendation_rate", "avg_rate"]]
            .to_json(orient="records", force_ascii=False)
        ),
        "top_brands_stats": json.loads(top_brands.reset_index().to_json(orient="records", force_ascii=False)),
        "brand_samples": brand_samples,
        "negative_comment_batches": negative_comment_batches,
    }

    return prep_data, product_stats, flagged


## بخش ۷ — اجرای تحلیل برای همه‌ی دسته‌های واجد شرایط

حلقه‌ی زیر تابع `analyze_category` را برای تک‌تک دسته‌های داخل
`TARGET_CATEGORIES` اجرا می‌کند و نتیجه را در `all_prep_data` جمع می‌کند.
دسته‌هایی که پس از فیلتر `MIN_COMMENTS_FOR_RATE` داده‌ی کافی نداشته باشند،
با یک پیام رد می‌شوند و در خروجی نهایی قرار نمی‌گیرند.


In [ ]:
all_prep_data = []
sample_plot_data = None  # برای رسم نمودار نمونه در بخش بعد نگه داشته می‌شود

for category_name in TARGET_CATEGORIES:
    category_products = df_products[df_products[CATEGORY_COLUMN] == category_name].copy()
    result = analyze_category(category_name, category_products, df_all_comments)

    if result is None:
        print(f"—  رد شد (داده‌ی کافی برای تحلیل نیست): {category_name}")
        continue

    prep_data, product_stats, flagged = result
    all_prep_data.append(prep_data)

    if sample_plot_data is None:
        sample_plot_data = (category_name, product_stats, flagged)

    print(f"OK {category_name} — محصولات تحلیل‌شده: {prep_data['products_analyzed']}، "
          f"محصولات پرنظر/نامطلوب: {len(prep_data['high_volume_low_recommendation_products'])}، "
          f"بچ‌های نظر منفی: {len(prep_data['negative_comment_batches'])}")

print(f"\nمجموع دسته‌های پردازش‌شده: {len(all_prep_data)} از {len(TARGET_CATEGORIES)} دسته‌ی واجد شرایط")


## بخش ۸ — نمودار نمونه (فقط برای بازبینی چشمی)

رسم نمودار برای همه‌ی دسته‌ها عملاً قابل استفاده نیست (ده‌ها نمودار پشت‌سرهم).
به‌جای آن، فقط برای **اولین دسته‌ای که با موفقیت تحلیل شده** (`sample_plot_data`)
دو نمودار نمونه رسم می‌شود تا صحت منطق تحلیل قابل بررسی باشد. این بخش صرفاً
جنبه‌ی بازبینی دارد و در فایل خروجی JSON ذخیره نمی‌شود.


In [ ]:
if sample_plot_data is not None:
    sample_category, sample_product_stats, sample_flagged = sample_plot_data

    plt.figure(figsize=(7, 5))
    plt.scatter(sample_product_stats["comment_count"], sample_product_stats["recommendation_rate"], alpha=0.5, label="All products")
    plt.scatter(sample_flagged["comment_count"], sample_flagged["recommendation_rate"], color="red", label="High-volume / Low-recommendation")
    plt.xlabel("Comment count")
    plt.ylabel("Recommendation rate")
    plt.title(f"Product Volume vs. Recommendation Rate — {sample_category} (sample)")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("هیچ دسته‌ای با موفقیت تحلیل نشد؛ نموداری برای رسم وجود ندارد.")


## بخش ۹ — بسته‌بندی و ذخیره‌ی `category_prep_data.json` (همه‌ی دسته‌ها)

این تنها فایلی است که باید از Kaggle دانلود کنید. حاوی هیچ کلید API یا
اطلاعات محرمانه‌ای نیست — فقط داده‌ی عمومی دیتاست است.

**نکته‌ی مهم درباره‌ی فرم خروجی:** اسکیمای هر آیتم دسته دقیقاً همانی است که
پیش‌تر برای یک دسته تولید می‌شد (`category`, `products_analyzed`,
`high_volume_low_recommendation_products`, `top_brands_stats`,
`brand_samples`, `negative_comment_batches`, ...) — **چیزی در آن تغییر
نکرده**. تنها تفاوت این است که حالا همه‌ی این آیتم‌ها داخل یک آرایه به نام
`categories` قرار می‌گیرند، چون به‌جای یک دسته، همه‌ی دسته‌های واجد شرایط
پردازش شده‌اند. نوت‌بوک بعدی (`category_analysis_local.ipynb`) باید روی
`data["categories"]` حلقه بزند و برای هر آیتم، همان منطق قبلی را (که برای
یک دسته نوشته بودید) اجرا کند.


In [ ]:
output_data = {
    "categories": all_prep_data,
}

with open(OUTPUT_PREP_JSON, "w", encoding="utf-8") as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

print(f"Saved: {OUTPUT_PREP_JSON}")
print(f"Categories included: {len(all_prep_data)}")
print("Download this file from Kaggle's output panel, then run category_analysis_local.ipynb next to it.")
